# Baseline Model: TF-IDF + Logistic Regression

## Objective

The objective of this notebook is to build a baseline machine learning model for telecom fault classification. The engineered `fault_description` feature will be converted into numerical vectors using TF-IDF Vectorization, and a Logistic Regression classifier will be trained to predict the corresponding fault category.

In [52]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib


In [53]:
df = pd.read_csv(r"C:\Users\athar\AI Projects\telecom-fault-rag-lora\data\processed\telecom_fault_dataset_model_ready.csv")

In [54]:
df.head()

,Unnamed: 0,incident_id,timestamp,network_element,region,raw_log,alarm_type,kpi_summary,packet_loss_percent,latency_ms,...,handover_failure_rate,prb_utilization,sinr_db,fault_category,root_cause,severity,resolution_steps,source_dataset,log_length,fault_description
0,0,INC_000001,2024-03-02 22:20:00,eNodeB_9741,Mumbai-West,- 1120266925 2005.07.01 R25-M1-N2-C:J10-U01 20...,ECC Memory Correction,"packet_loss=2.17%, latency=130.72ms, call_drop...",2.17,130.72,...,0.57,66.47,17.94,Memory/Cache Error,Processor cache parity fault,High,Review ECC counters and node health; escalate ...,Loghub-BGL,149,Raw Log: - 1120266925 2005.07.01 R25-M1-N2-C:J...
1,1,INC_000002,2024-03-06 04:20:00,AMF_5916,Gurugram-Sector-44,BLOCK* NameSystem.delete: blk_3280859030919654...,Missing Block Mapping,"packet_loss=3.63%, latency=62.71ms, call_drop_...",3.63,62.71,...,5.84,50.11,11.70,Configuration Error,Incorrect or missing network mapping metadata,High,Audit recent configuration changes and rollbac...,Loghub-HDFS,96,Raw Log: BLOCK* NameSystem.delete: blk_3280859...
2,2,INC_000003,2024-03-01 08:48:00,gNodeB_2833,Hyderabad-HITEC-City,- 1120259941 2005.07.01 R25-M1-N2-C:J08-U01 20...,Memory Parity Alarm,"packet_loss=2.2%, latency=105.66ms, call_drop_...",2.20,105.66,...,2.92,50.80,18.63,Memory/Cache Error,Correctable memory or cache parity error,High,"Monitor repeated parity errors, check memory d...",Loghub-BGL,149,Raw Log: - 1120259941 2005.07.01 R25-M1-N2-C:J...
3,3,INC_000004,2024-03-07 16:30:00,gNodeB_7671,Chennai-OMR,Receiving block blk_3598226258881465924 src: /...,Block Transfer Latency,"packet_loss=4.21%, latency=138.93ms, call_drop...",4.21,138.93,...,3.52,71.62,14.99,Transport Degradation,Intermittent degradation in node-to-node data ...,High,Monitor transfer delay and escalate if degrada...,Loghub-HDFS,94,Raw Log: Receiving block blk_35982262588814659...
4,4,INC_000005,2024-03-03 18:00:00,UPF_9177,Pune-Hinjewadi,BLOCK* NameSystem.delete: blk_-438534717113603...,Configuration Consistency Alarm,"packet_loss=3.74%, latency=77.99ms, call_drop_...",3.74,77.99,...,5.73,82.98,17.14,Configuration Error,Incorrect or missing network mapping metadata,High,Check replication policy and restore missing m...,Loghub-HDFS,96,Raw Log: BLOCK* NameSystem.delete: blk_-438534...


In [55]:
X = df["fault_description"]
y = df["fault_category"]

In [56]:
X.head()

0    Raw Log: - 1120266925 2005.07.01 R25-M1-N2-C:J...
1    Raw Log: BLOCK* NameSystem.delete: blk_3280859...
2    Raw Log: - 1120259941 2005.07.01 R25-M1-N2-C:J...
3    Raw Log: Receiving block blk_35982262588814659...
4    Raw Log: BLOCK* NameSystem.delete: blk_-438534...
Name: fault_description, dtype: str

In [57]:
y.head()

0       Memory/Cache Error
1      Configuration Error
2       Memory/Cache Error
3    Transport Degradation
4      Configuration Error
Name: fault_category, dtype: str

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [59]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (3200,)
X_test shape: (800,)
y_train shape: (3200,)
y_test shape: (800,)


In [60]:
# Build baseline pipeline
baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])


In [61]:

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)

In [62]:

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
matrix = confusion_matrix(y_test, y_pred)

print("\nBaseline Accuracy:", accuracy)
print("\nClassification Report:\n")
print(report)



Baseline Accuracy: 0.9975

Classification Report:

                       precision    recall  f1-score   support

    Application Crash       1.00      1.00      1.00       200
       Backhaul Issue       1.00      1.00      1.00        74
  Configuration Error       1.00      1.00      1.00        45
   Core Network Issue       1.00      1.00      1.00        12
         Interference       1.00      1.00      1.00        14
   Memory/Cache Error       1.00      0.99      0.99        87
               Normal       0.99      1.00      1.00       117
         Power Outage       0.00      0.00      0.00         1
Transport Degradation       1.00      1.00      1.00       250

             accuracy                           1.00       800
            macro avg       0.89      0.89      0.89       800
         weighted avg       1.00      1.00      1.00       800



c:\Users\athar\AI Projects\telecom-fault-rag-lora\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\athar\AI Projects\telecom-fault-rag-lora\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\athar\AI Projects\telecom-fault-rag-lora\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(